In [1]:
import osmnx as ox
import networkx as nx
import geopandas as gpd
import warnings


from graphs.speeds import kmh_to_mm, set_graph_travel_times
from graphs.settings import DEFAULT_SPEEDS

In [2]:
# Загрузка графа и полигона границ
G = ox.load_graphml("tests/data/test_rng.ml")
area = gpd.read_file("tests/data/test_polygon.gpkg")

In [3]:
# %%time


print(DEFAULT_SPEEDS)
speeds = [kmh_to_mm(s) for s in DEFAULT_SPEEDS]
print(speeds)
print('='*80)

# def set_graph_travel_times(G: nx.MultiDiGraph,
#                      speeds: list = DEFAULT_SPEEDS,
#                      morph_function: callable = None,
#                      travel_time_field: str = 'travel_time',
#                      speed_field: str = 'maxspeed',
#                      ):
#     """
#     Добавление скоростей в граф

#     Аргументы
#     ---------
#     `G`: networkx.MultiDiGraph
#         Граф улично-дорожной сети
#     `speeds`: dict
#         Список скоростей для 5 классов дорог:
#         ```
#         1 - Магистральные городские дороги и улицы общегородского значения
#         2 - Магистральные улицы районного значения
#         3 - Улицы и дороги местного значения
#         4 - Служебные проезды: внутриквартальные, въездные, парковочные и т.д.
#         5 - Пешеходные зоны и территории не являющиеся проезжей частью,
#             но теоретически пригодные для передвижения пожарных автомобилей
#         ```
#     `morph_function`: callable
#         Функция преобразования скоростей.
#         Если указана, то будет применена к `speeds` как `morph_function(speeds)`
#         Может быть использована для перевода `километры в час` -> `метры в минуту`
#     `travel_time_field`: str = 'travel_time'
#         Название поля в котором будет сохранено время следования
#     `speed_field`: str ='maxspeed'
#         Название поля в котором будет сохранено значение скорости следования

#     Возвращает
#     ----------
#     None
#     """

#     # 0. Проверка входящих данных
#     if not isinstance(G, nx.MultiDiGraph):
#         raise TypeError('Тип данных аргумента `env` должен быть nx.MultiDiGraph')
#     if not isinstance(speeds, list):
#         raise ValueError('Тип данных аргумента `speeds` должен быть список')
#     if len(speeds) != 5:
#         raise ValueError('Аргумент `speeds` должен содержать строго 5 элементов!')

#     if G.graph.get('simplified', False):
#         warnings.warn("Граф был ранее упрощен! Оценка времени следования такого графа может повлечь неточности. " + \
#               "Рекомендуется устанавливать время следования для неупрощенного графа " + \
#               "и только после этого упрощать")

#     # 1 Перевод скоростей в некоторые значения
#     if morph_function is None:
#         s1, s2, s3, s4, s5 = speeds
#     else:
#         s1, s2, s3, s4, s5 = [morph_function(kmh) for kmh in speeds]

#     # 2 Установка скоростей для тегов OSM
#     sp = {
#             "trunk":          s1,           # Важные дороги, не являющиеся автомагистралями
#             "trunk_link":     s1,           # Важные дороги, не являющиеся автомагистралями
#             "motorway":       s1,           # Автомагистрали 
#             "motorway_link":  s1,           # Автомагистрали 

#             "primary":        s2,           # Автомобильные дороги регионального значения
#             "primary_link":   s2,           # Автомобильные дороги регионального значения
#             "secondary":      s2,           # Автомобильные дороги областного значения
#             "secondary_link": s2,           # Автомобильные дороги областного значения
#             "unclassified":   s2,           # Остальные автомобильные дороги местного значения, образующие соединительную сеть дорог.

#             "tertiary":       s3,           # Более важные автомобильные дороги среди прочих автомобильных дорог местного значения
#             "tertiary_link":  s3,           # Более важные автомобильные дороги среди прочих автомобильных дорог местного значения
#             "residential":    s3,           # Дороги, которые проходят внутри жилых зон, а также используются для подъезда к ним
#             "living_street":  s3,           # Жилые зоны и дворовые проезды

#             "road":           s4,           # Линии, возможно, являющиеся дорогами. Временный тег, которым следует помечать линии до уточнения.
#             "service":        s4,           # Служебные проезды: внутриквартальные, въездные, парковочные и т.д.
#             "track":          s4,           # Дороги сельскохозяйственного назначения, лесные дороги, не ведущие к жилым или промышленным объектам, неофициальные грунтовки, козьи тропы

#             "footway":        s5,           # Пешеходные дорожки, тротуары. 
#             "path":           s5,           # Тропа (чаще всего, стихийная) использующаяся пешеходами, либо одним или несколькими видами транспорта, кроме четырехколесного (лыжи, снегоход, велосипед).
#             "pedestrian":     s5,           # Для обозначения улиц городов (такого же класса как residential), выделенных для пешеходов.

#             "steps":          s5,           # Лестницы, лестничные пролёты. 
#             "cycleway":       s5,           # Велодорожка, обозначенная соответствующим дорожным знаком. 
#             "bridleway":      s5,           # Дорожки для верховой езды.
#             "corridor":       s5,           # Коридоры внутри крупных зданий
#         }

#     # 3. расчет и установка времени следования и скоростей
#     attributes = {}
#     for edge in G.edges:
#         road = G.get_edge_data(*edge).get('highway', 'other')
#         length = G.get_edge_data(*edge).get('length')
#         if isinstance(road, list):
#             road_speeds = [sp.get(rf, s5) for rf in road]
#             speed       = sum(road_speeds) / len(road_speeds)
#         else:
#             speed  = sp.get(road, s5)

#         weight = length/speed
#         attributes[*edge] = {travel_time_field: weight, speed_field: speed}

#     nx.set_edge_attributes(G, attributes)



set_graph_travel_times(G, speeds)
try:
    G = ox.simplify_graph(G)
except ox._errors.GraphSimplificationError:
    print('Граф уже упрощен')

[49, 37, 26, 16, 5]
[816.67, 616.67, 433.33, 266.67, 83.33]


In [5]:
edges = ox.graph_to_gdfs(G, nodes=False)
edges[:5]

osmid  oneway lanes      ref    highway  \
u         v          key                                                
290700365 290700367  0    111848408   False     2  04Н-374  secondary   
          4796070877 0    111848408   False     2  04Н-374  secondary   
          2034402054 0    192901371   False   NaN      NaN    service   
290700367 4329379203 0    789781430   False     2  04Н-374  secondary   
          2034401903 0     73258820   False     2      NaN   tertiary   

                         maxspeed reversed   length  travel_time  \
u         v          key                                           
290700365 290700367  0     616.67    False  211.139     0.342386   
          4796070877 0     616.67     True  238.064     0.386048   
          2034402054 0     266.67     True  140.574     0.527146   
290700367 4329379203 0     616.67    False  156.978     0.254558   
          2034401903 0     433.33    False  175.718     0.405506   

                                                                   geometry  \
u         v          key                                                      
290700365 290700367  0    LINESTRING (93.34714 56.15506, 93.34673 56.154...   
          4796070877 0    LINESTRING (93.34714 56.15506, 93.34753 56.155...   
          2034402054 0    LINESTRING (93.34714 56.15506, 93.34633 56.155...   
290700367 4329379203 0    LINESTRING (93.34631 56.15322, 93.34583 56.15183)   
          2034401903 0    LINESTRING (93.34631 56.15322, 93.34682 56.153...   

                         name service bridge tunnel access  
u         v          key                                    
290700365 290700367  0    NaN     NaN    NaN    NaN    NaN  
          4796070877 0    NaN     NaN    NaN    NaN    NaN  
          2034402054 0    NaN     NaN    NaN    NaN    NaN  
290700367 4329379203 0    NaN     NaN    NaN    NaN    NaN  
          2034401903 0    NaN     NaN    NaN    NaN    NaN